# Drug Response Prediction on GDSC2 — a Karpathy-style progression

Each section is runnable end-to-end and **directly comparable** to the one before it.
The rule of the game: *every model must beat the per-drug-mean baseline on a cell-line-blind split.* Most don't, at first. That's the point.

**Data**
- `GDSC2_fitted_dose_response_27Oct23.xlsx` — labels (`LN_IC50`)
- `model_list_20260420.csv` — cell-line metadata (tissue)
- figshare `43146652` — bulk RNA-seq expression matrix (rename to `rnaseq_tpm.csv`)

All three join on `SANGER_MODEL_ID`. If the files aren't present, the notebook **falls back to synthetic data** so every cell runs; swap in real paths in the CONFIG cell.

**Environment note:** stages 1–3 need only numpy/pandas/sklearn/scipy. Stage 3 uses PyTorch if present (else an sklearn MLP). Stage 4 needs `torch` + `rdkit` — run it on your own machine (your RTX box is ideal); it's gated so the notebook won't crash without them.

## Stage 0 — Config, loading, join, splits, metrics

In [ ]:
"""
Stages 0-1: load, join, baselines. Validated standalone before notebook assembly.
"""
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# ----------------------------------------------------------------------------
# CONFIG — point these at your downloads. If a file is missing, the notebook
# falls back to synthetic data so every cell still runs.
# ----------------------------------------------------------------------------
PATHS = {
    "response": "GDSC2_fitted_dose_response_27Oct23.xlsx",   # the labels
    "model_list": "model_list_20260420.csv",                  # cell-line metadata
    "expression": "rnaseq_tpm.csv",                           # figshare 43146652 (rename after download)
}
RANDOM_STATE = 1337
N_TOP_GENES = 1000   # most-variable genes kept as features


def _resolve(df, candidates, required=True, what=""):
    """Return the first column in `candidates` present in df (case-insensitive)."""
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    if required:
        raise KeyError(f"None of {candidates} found for {what}. Have: {list(df.columns)[:20]}")
    return None


def make_synthetic(n_lines=120, n_drugs=40, n_genes=1500, seed=RANDOM_STATE):
    """A small synthetic stand-in with the same shape/semantics as the real join.
    Signal is deliberately weak-but-real so baselines are hard to beat (as in reality).
    """
    rng = np.random.default_rng(seed)
    line_ids = [f"SIDM{i:05d}" for i in range(n_lines)]
    tissues = rng.choice(["Lung", "Breast", "Skin", "Colon", "Blood"], size=n_lines)
    genes = [f"GENE{i}" for i in range(n_genes)]

    # expression: genes x lines, a few genes carry tissue structure
    expr = rng.normal(0, 1, size=(n_genes, n_lines)).astype(np.float32)
    for t_idx, t in enumerate(np.unique(tissues)):
        mask = tissues == t
        expr[t_idx * 10:(t_idx + 1) * 10][:, mask] += 2.0
    expr_df = pd.DataFrame(expr, index=genes, columns=line_ids)

    drug_ids = np.arange(1001, 1001 + n_drugs)
    # latent factors that genuinely link expression -> response
    W = rng.normal(0, 1, size=(n_genes, 4)) * (rng.random((n_genes, 1)) < 0.05)
    line_lat = (expr.T @ W)                       # n_lines x 4
    drug_lat = rng.normal(0, 1, size=(n_drugs, 4))
    drug_base = rng.normal(2.0, 1.5, size=n_drugs)  # per-drug mean dominates (realistic)

    rows = []
    for li, lid in enumerate(line_ids):
        for di, did in enumerate(drug_ids):
            signal = 0.15 * float(line_lat[li] @ drug_lat[di])
            y = drug_base[di] + signal + rng.normal(0, 0.7)
            rows.append((lid, tissues[li], int(did), f"DRUG{di}", y))
    resp = pd.DataFrame(rows, columns=["SANGER_MODEL_ID", "tissue", "DRUG_ID", "DRUG_NAME", "LN_IC50"])
    return resp, expr_df


def load_real():
    """Attempt to load the three real files. Returns (resp, expr_df) or raises."""
    import os
    for k, p in PATHS.items():
        if not os.path.exists(p):
            raise FileNotFoundError(p)

    resp = pd.read_excel(PATHS["response"])
    cid = _resolve(resp, ["SANGER_MODEL_ID", "COSMIC_ID", "CELL_LINE_NAME"], what="cell id")
    drug = _resolve(resp, ["DRUG_ID"], what="drug id")
    dname = _resolve(resp, ["DRUG_NAME"], required=False, what="drug name")
    y = _resolve(resp, ["LN_IC50"], what="label")
    keep = {cid: "SANGER_MODEL_ID", drug: "DRUG_ID", y: "LN_IC50"}
    if dname:
        keep[dname] = "DRUG_NAME"
    resp = resp.rename(columns=keep)[list(keep.values())].copy()

    models = pd.read_csv(PATHS["model_list"], low_memory=False)
    mid = _resolve(models, ["model_id", "SANGER_MODEL_ID"], what="model id")
    tis = _resolve(models, ["tissue", "cancer_type", "tissue_status"], what="tissue")
    models = models.rename(columns={mid: "SANGER_MODEL_ID", tis: "tissue"})[["SANGER_MODEL_ID", "tissue"]]
    resp = resp.merge(models, on="SANGER_MODEL_ID", how="left")

    # expression: genes x samples (typical Cell Model Passports orientation). First col = gene id.
    expr_df = pd.read_csv(PATHS["expression"], index_col=0)
    # keep only columns (samples) that are model ids appearing in resp
    common = [c for c in expr_df.columns if c in set(resp["SANGER_MODEL_ID"])]
    if len(common) < 0.3 * resp["SANGER_MODEL_ID"].nunique():
        # likely transposed: samples x genes
        expr_df = expr_df.T
        common = [c for c in expr_df.columns if c in set(resp["SANGER_MODEL_ID"])]
    expr_df = expr_df[common]
    return resp, expr_df


def get_data():
    try:
        resp, expr_df = load_real()
        print("Loaded REAL data.")
    except (FileNotFoundError, KeyError) as e:
        print(f"Real data unavailable ({type(e).__name__}: {e}). Using SYNTHETIC fallback.")
        resp, expr_df = make_synthetic()
    # drop rows we can't use, restrict to lines that have expression
    resp = resp.dropna(subset=["LN_IC50", "SANGER_MODEL_ID", "DRUG_ID"])
    lines_with_expr = set(expr_df.columns)
    resp = resp[resp["SANGER_MODEL_ID"].isin(lines_with_expr)].reset_index(drop=True)
    if "tissue" not in resp or resp["tissue"].isna().all():
        resp["tissue"] = "UNKNOWN"
    resp["tissue"] = resp["tissue"].fillna("UNKNOWN")
    return resp, expr_df


# ----------------------------------------------------------------------------
# SPLITS — three regimes. Cell-line-blind is the one that matches the goal.
# ----------------------------------------------------------------------------
def make_split(resp, regime="cell_line", test_frac=0.2, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(resp))
    if regime == "pair":
        rng.shuffle(idx)
        cut = int(len(idx) * (1 - test_frac))
        return idx[:cut], idx[cut:]
    key = "SANGER_MODEL_ID" if regime == "cell_line" else "DRUG_ID"
    groups = resp[key].unique()
    order = rng.permutation(len(groups))
    groups = groups[order]
    cut = int(len(groups) * (1 - test_frac))
    train_groups = set(groups[:cut])
    tr = resp.index[resp[key].isin(train_groups)].to_numpy()
    te = resp.index[~resp[key].isin(train_groups)].to_numpy()
    return tr, te


def metrics(y_true, y_pred):
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    r = float(pearsonr(y_true, y_pred)[0]) if len(y_true) > 2 else float("nan")
    return rmse, r


# ----------------------------------------------------------------------------
# STAGE 1: no-ML baselines
# ----------------------------------------------------------------------------
def baseline_per_drug(resp, tr, te):
    drug_mean = resp.iloc[tr].groupby("DRUG_ID")["LN_IC50"].mean()
    glob = resp.iloc[tr]["LN_IC50"].mean()
    pred = resp.iloc[te]["DRUG_ID"].map(drug_mean).fillna(glob).to_numpy()
    return metrics(resp.iloc[te]["LN_IC50"].to_numpy(), pred)


def baseline_per_drug_tissue(resp, tr, te):
    dt_mean = resp.iloc[tr].groupby(["DRUG_ID", "tissue"])["LN_IC50"].mean()
    drug_mean = resp.iloc[tr].groupby("DRUG_ID")["LN_IC50"].mean()
    glob = resp.iloc[tr]["LN_IC50"].mean()

    def lookup(row):
        try:
            return dt_mean.loc[(row["DRUG_ID"], row["tissue"])]
        except KeyError:
            return drug_mean.get(row["DRUG_ID"], glob)

    pred = resp.iloc[te].apply(lookup, axis=1).to_numpy()
    return metrics(resp.iloc[te]["LN_IC50"].to_numpy(), pred)

### Load the data and look at it
Always eyeball shapes and the label distribution before modeling.

In [ ]:
resp, expr_df = get_data()
print(f'response rows: {len(resp):,} | lines: {resp.SANGER_MODEL_ID.nunique()} '
      f'| drugs: {resp.DRUG_ID.nunique()} | genes: {expr_df.shape[0]}')
print(f'LN_IC50  mean={resp.LN_IC50.mean():.3f}  std={resp.LN_IC50.std():.3f}')
resp.head()

## Stage 1 — No-ML baselines

Predict the per-drug mean `LN_IC50`. Then per-(drug, tissue) mean. These RMSE / Pearson-r numbers are the bar. Note the three split regimes:
- **pair**: random rows — easy, leaks cell-line and drug identity into test.
- **cell_line**: held-out cell lines — *this is your stated goal* (predict an unseen line).
- **drug**: held-out drugs — per-drug mean is undefined for unseen drugs (→ flat prediction). This is *why* Stage 4 needs drug features.

In [ ]:
for regime in ['pair', 'cell_line', 'drug']:
    tr, te = make_split(resp, regime)
    r1 = baseline_per_drug(resp, tr, te)
    r2 = baseline_per_drug_tissue(resp, tr, te)
    print(f'[{regime:9s}] per-drug RMSE={r1[0]:.3f} r={r1[1]:.3f}   '
          f'per-drug+tissue RMSE={r2[0]:.3f} r={r2[1]:.3f}')

## Stage 2 — ElasticNet on `[expression ⊕ drug-onehot]`

First evidence that expression carries signal beyond the per-drug mean. Feature notes:
- Top-`N_TOP_GENES` most-variable genes, selected **on train lines only**.
- Z-scored using **train** mean/std (no leakage).
- Drug identity as one-hot fit on train drugs; unseen drugs become an all-zero block.

In [ ]:
# ----------------------------------------------------------------------------
# FEATURES: top-variance genes (z-scored on TRAIN) + drug one-hot
# Standardization stats are computed on TRAIN only — no leakage.
# ----------------------------------------------------------------------------
def select_top_genes(expr_df, tr_lines, n_top=N_TOP_GENES):
    sub = expr_df[list(tr_lines)]
    var = sub.var(axis=1)
    return var.sort_values(ascending=False).head(min(n_top, len(var))).index.tolist()


def build_features(resp, expr_df, tr, te, n_top=N_TOP_GENES):
    tr_lines = resp.iloc[tr]["SANGER_MODEL_ID"].unique()
    top_genes = select_top_genes(expr_df, tr_lines, n_top)

    # gene matrix: lines x genes, standardized on train lines
    G = expr_df.loc[top_genes].T          # samples x genes
    mu = G.loc[list(tr_lines)].mean(axis=0)
    sd = G.loc[list(tr_lines)].std(axis=0).replace(0, 1.0)
    G = (G - mu) / sd

    # drug one-hot fit on train drugs only; unseen drugs -> all-zero row
    train_drugs = sorted(resp.iloc[tr]["DRUG_ID"].unique())
    drug_index = {d: i for i, d in enumerate(train_drugs)}
    D = np.zeros((len(resp), len(train_drugs)), dtype=np.float32)
    for row_i, d in enumerate(resp["DRUG_ID"].to_numpy()):
        j = drug_index.get(d)
        if j is not None:
            D[row_i, j] = 1.0

    Gmat = G.loc[resp["SANGER_MODEL_ID"].to_numpy()].to_numpy(dtype=np.float32)
    X = np.hstack([Gmat, D])
    y = resp["LN_IC50"].to_numpy(dtype=np.float32)
    return X, y, len(top_genes), len(train_drugs)


# ----------------------------------------------------------------------------
# STAGE 2: ElasticNet
# ----------------------------------------------------------------------------
def run_elasticnet(X, y, tr, te):
    from sklearn.linear_model import ElasticNet
    m = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE)
    m.fit(X[tr], y[tr])
    return metrics(y[te], m.predict(X[te]))


# ----------------------------------------------------------------------------
# STAGE 3: MLP — torch if available, else sklearn MLPRegressor
# ----------------------------------------------------------------------------
def run_mlp(X, y, tr, te, epochs=60):
    try:
        import torch
        import torch.nn as nn
        torch.manual_seed(RANDOM_STATE)
        dev = "cuda" if torch.cuda.is_available() else "cpu"

        # standardize y on train for stable optimization
        ymu, ysd = y[tr].mean(), y[tr].std() + 1e-8
        Xtr = torch.tensor(X[tr], device=dev)
        ytr = torch.tensor((y[tr] - ymu) / ysd, device=dev).unsqueeze(1)
        Xte = torch.tensor(X[te], device=dev)

        net = nn.Sequential(
            nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        ).to(dev)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
        loss_fn = nn.MSELoss()
        bs = 512
        n = Xtr.shape[0]
        for ep in range(epochs):
            net.train()
            perm = torch.randperm(n, device=dev)
            for i in range(0, n, bs):
                b = perm[i:i + bs]
                opt.zero_grad()
                loss = loss_fn(net(Xtr[b]), ytr[b])
                loss.backward()
                opt.step()
        net.eval()
        with torch.no_grad():
            pred = net(Xte).cpu().numpy().ravel() * ysd + ymu
        return metrics(y[te], pred), "torch"
    except ImportError:
        from sklearn.neural_network import MLPRegressor
        m = MLPRegressor(hidden_layer_sizes=(256, 64), activation="relu",
                         alpha=1e-4, max_iter=epochs, random_state=RANDOM_STATE,
                         early_stopping=True)
        m.fit(X[tr], y[tr])
        return metrics(y[te], m.predict(X[te])), "sklearn"

In [ ]:
tr, te = make_split(resp, 'cell_line')
X, y, ng, nd = build_features(resp, expr_df, tr, te)
print(f'feature matrix: {X.shape}  ({ng} genes + {nd} drug-onehot)')
base = baseline_per_drug(resp, tr, te)
en = run_elasticnet(X, y, tr, te)
print(f'baseline per-drug  RMSE={base[0]:.3f} r={base[1]:.3f}')
print(f'ElasticNet         RMSE={en[0]:.3f} r={en[1]:.3f}')

## Stage 3 — MLP (2–3 layers, ReLU, dropout)

Same feature vector as Stage 2, so the comparison is clean: does nonlinearity help? Uses PyTorch if available (GPU-aware), else falls back to sklearn's `MLPRegressor`. Run this across all three regimes — watch the `cell_line` and `drug` columns, not `pair`.

In [ ]:
for regime in ['pair', 'cell_line', 'drug']:
    tr, te = make_split(resp, regime)
    X, y, ng, nd = build_features(resp, expr_df, tr, te)
    base = baseline_per_drug(resp, tr, te)
    en = run_elasticnet(X, y, tr, te)
    (mlp, backend) = run_mlp(X, y, tr, te)
    print(f'[{regime:9s}] base RMSE={base[0]:.3f} r={base[1]:.3f} | '
          f'EN RMSE={en[0]:.3f} r={en[1]:.3f} | '
          f'MLP({backend}) RMSE={mlp[0]:.3f} r={mlp[1]:.3f}')

## Stage 4 — Two-tower: cell-line encoder × drug encoder

The architecturally interesting jump. A **cell-line tower** maps expression → embedding; a **drug tower** maps a Morgan fingerprint (from SMILES) → embedding; the two are combined and passed to an MLP head. This is what lets the model **generalize to held-out drugs**, because the drug is now described by its chemistry rather than a one-hot id.

**Requirements:** `torch` and `rdkit`. The cell below checks for them and **skips gracefully** if absent. To get drug SMILES, map `DRUG_NAME`/`DRUG_ID` to PubChem CIDs (e.g. via the GDSC compound annotation file or PubChem) and compute fingerprints with RDKit. The synthetic fallback fabricates random fingerprints just so the architecture runs.

In [ ]:
def get_drug_fingerprints(resp, n_bits=1024):
    """Return {DRUG_ID: np.array(n_bits)}.
    Real path: map DRUG_ID -> SMILES, then RDKit Morgan fingerprint.
    Fallback: deterministic random bits per drug (architecture test only)."""
    try:
        from rdkit import Chem
        from rdkit.Chem import AllChem
        smiles_map = load_drug_smiles(resp)  # you implement: DRUG_ID -> SMILES str
        fps = {}
        for did, smi in smiles_map.items():
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            bv = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
            arr = np.zeros((n_bits,), dtype=np.float32)
            from rdkit.DataStructs import ConvertToNumpyArray
            ConvertToNumpyArray(bv, arr)
            fps[did] = arr
        return fps
    except (ImportError, NameError):
        rng = np.random.default_rng(RANDOM_STATE)
        return {d: (rng.random(n_bits) < 0.1).astype(np.float32)
                for d in resp['DRUG_ID'].unique()}


def run_two_tower(resp, expr_df, regime='drug', epochs=80, n_bits=1024):
    try:
        import torch, torch.nn as nn
    except ImportError:
        print('PyTorch not installed — skipping Stage 4. Run on your own machine.')
        return None
    torch.manual_seed(RANDOM_STATE)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tr, te = make_split(resp, regime)

    # cell-line features: top genes, standardized on train
    tr_lines = resp.iloc[tr]['SANGER_MODEL_ID'].unique()
    genes = select_top_genes(expr_df, tr_lines, N_TOP_GENES)
    G = expr_df.loc[genes].T
    mu, sd = G.loc[list(tr_lines)].mean(axis=0), G.loc[list(tr_lines)].std(axis=0).replace(0, 1)
    G = (G - mu) / sd

    fps = get_drug_fingerprints(resp, n_bits)
    Gmat = G.loc[resp['SANGER_MODEL_ID'].to_numpy()].to_numpy(np.float32)
    Dmat = np.stack([fps[d] for d in resp['DRUG_ID'].to_numpy()])
    y = resp['LN_IC50'].to_numpy(np.float32)
    ymu, ysd = y[tr].mean(), y[tr].std() + 1e-8

    Gt = torch.tensor(Gmat, device=dev); Dt = torch.tensor(Dmat, device=dev)
    yt = torch.tensor((y - ymu) / ysd, device=dev).unsqueeze(1)
    tr_t = torch.tensor(tr, device=dev); te_t = torch.tensor(te, device=dev)

    class TwoTower(nn.Module):
        def __init__(self, n_gene, n_fp, emb=64):
            super().__init__()
            self.cell = nn.Sequential(nn.Linear(n_gene, 256), nn.ReLU(),
                                      nn.Dropout(0.3), nn.Linear(256, emb), nn.ReLU())
            self.drug = nn.Sequential(nn.Linear(n_fp, 256), nn.ReLU(),
                                      nn.Dropout(0.3), nn.Linear(256, emb), nn.ReLU())
            self.head = nn.Sequential(nn.Linear(2 * emb, 64), nn.ReLU(),
                                      nn.Dropout(0.3), nn.Linear(64, 1))
        def forward(self, g, d):
            return self.head(torch.cat([self.cell(g), self.drug(d)], dim=1))

    net = TwoTower(Gmat.shape[1], n_bits).to(dev)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.MSELoss(); bs = 512
    for ep in range(epochs):
        net.train(); perm = tr_t[torch.randperm(len(tr_t), device=dev)]
        for i in range(0, len(perm), bs):
            b = perm[i:i+bs]; opt.zero_grad()
            loss = loss_fn(net(Gt[b], Dt[b]), yt[b]); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        pred = net(Gt[te_t], Dt[te_t]).cpu().numpy().ravel() * ysd + ymu
    rmse, r = metrics(y[te], pred)
    print(f'[{regime}] TwoTower  RMSE={rmse:.3f} r={r:.3f}  (vs baseline below)')
    print(f'[{regime}] baseline  RMSE={baseline_per_drug(resp, tr, te)[0]:.3f}')
    return net

_ = run_two_tower(resp, expr_df, regime='drug')

## Stage 5 — Where to go next (sketches, not yet wired)

- **Graph drug encoder (GNN):** replace the fingerprint tower with a message-passing GNN over the molecular graph (atoms=nodes, bonds=edges). Usually beats fingerprints on held-out drugs.
- **Attention over genes:** let the cell tower attend to pathway-grouped genes instead of a flat MLP.
- **Multi-task head:** predict `LN_IC50` and `AUC` jointly with two output heads + summed loss; the shared trunk regularizes both.
- **Domain shift to patient-derived data:** GDSC lines ≠ PDX/PDO. Validate/fine-tune on a PDX pharmacogenomic set (e.g. Novartis PDXE) before trusting any clinical-flavored claim.

### How to read your results
Track one table: rows = models, columns = (pair / cell_line / drug) × (RMSE, r). A model only 'counts' if it beats the per-drug-mean baseline **in the cell_line column** (your goal) — and fingerprints/GNN earn their keep specifically in the **drug** column.